# Method 1 - Self-Organising Map (SOM)

Trains a topology-preserving SOM on the **full 219-band** EnMAP data (no
dimensionality reduction) and clusters pixels by winning neuron.

**What changed vs the original notebook**
- Preprocessing, seeding and post-processing now come from the shared `src/`
  package, so this method uses *exactly* the same valid pixels/bands as the
  k-means methods (previously the stacked-AE used a different fill mask).
- The SOM label map now receives the **same majority filter** as every other
  method (it was previously left unfiltered, making the comparison unfair).
- Intrinsic metrics are computed on the real 219-D feature space (the old code
  saved 2-D winner coordinates, which made Silhouette/CH/DB meaningless).
- All RNGs (numpy / torch / MiniSom) are seeded, so runs are reproducible.

In [ ]:
# Make the shared src/ package importable regardless of CWD.
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.seeding import seed_everything
from src.preprocessing import load_and_preprocess_enmap
from src.results import finalize_method
from src.postprocess import majority_filter

seed_everything()          # seed numpy / random / torch from config.SEED
print(f"Seed = {config.SEED}")

In [ ]:
# Canonical preprocessing (SOM runs on unstandardised reflectance, as in the paper).
res = load_and_preprocess_enmap(config.ENMAP_TIF, standardize=False)
X = res.data_valid
print("Training matrix:", X.shape)

In [ ]:
from src.pipeline import train_som
from src.som_metrics import describe_topographic_error

# Train the final SOM via the shared pipeline, which uses the CORRECTED
# training (one weight update per pixel per epoch, random order).
#
# The original notebook called `som.train_batch(X, num_iteration=1000)`.
# MiniSom builds its sample order as `arange(num_iterations) % len(data)` and
# train_batch does not shuffle, so that trained on pixels [0..999] only -
# 0.096% of the image, all from a patch in the top ~20 rows of the scene.
labels_raw, _, info, som = train_som(X, seed=config.SEED, verbose=True)

print(f"\nclusters: {info['n_clusters']}  |  updates: {info['updates']:,}")
print(f"Quantization Error : {info['Quantization Error']:.4f}")
print(f"Topographic Error  : {info['Topographic Error']:.4f} "
      f"-> {describe_topographic_error(info['Topographic Error'])}")

In [ ]:
# Post-process (majority filter), score intrinsic quality on 219-D space, save.
som_map = finalize_method("som", res, labels_raw, features_for_intrinsic=X, extra=info)
np.save(config.OUTPUTS_DIR / "som_weights.npy", som.get_weights())

def plot_cluster_map(labels_spatial, extent, title):
    n = int(np.nanmax(labels_spatial)) + 1 if np.isfinite(np.nanmax(labels_spatial)) else 1
    fig, ax = plt.subplots(figsize=(12, 9))
    cmap = plt.get_cmap("rainbow", n).copy()
    cmap.set_bad(color="white")
    im = ax.imshow(labels_spatial, cmap=cmap, extent=extent, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax, label="Cluster", fraction=0.046)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    return fig, ax

plot_cluster_map(som_map, res.extent, "SOM (9 clusters) - majority filtered")
plt.savefig(config.FIGURES_DIR / "som_classification.png", dpi=200, bbox_inches="tight")
plt.show()

## SOM hyper-parameter tuning

The paper chose sigma=0.5 / lr=0.1 by *visual* inspection of cluster maps, while
the SOM was training on 1,000 pixels from one corner. Both the criterion and the
model were broken, so those settings carry no evidence.

`run_som_tuning.py` re-tunes quantitatively with correct training, scoring each
setting on the SOM's own metrics (quantization / topographic error) and on
Kanimblan recall. Run it from a terminal:

```
.venv/Scripts/python.exe run_som_tuning.py       # -> Figures/som_tuning.csv
.venv/Scripts/python.exe run_som_training_fix.py # -> before/after the bug fix
```

Result: **sigma=0.5, 1 full epoch** beats the paper's setting on every metric
(QE 0.301 vs 0.388, TE 0.0038 vs 0.572, Kanimblan recall 0.766 vs 0.000).
Larger sigma gives near-perfect topology but zero Kanimblan recall - the
neighbourhood drags all nine neurons onto the dominant CSP spectra.